In [1]:
import numpy as np

# ============================================================
# PRECISION CONTROL
# ============================================================
DT = np.longdouble


def asDT(x):
    return np.asarray(x, dtype=DT)

# ============================================================
# TOTAL MQMQA INTEGRATION TEST (ternary-capable toy)
#   - 3 cations: A,B,C
#   - 2 anions: X,Y  => anion pairs XX,XY,YY
#   - 18 quadruplets total
#
# Verifies: H_tot_analytic = H_id_analytic + H_ex_analytic
#           H_tot_FD       = FD Hessian of G_tot = G_id + G_ex
#
# Flip ONLY case_map to cover ternary cases: "gamma", "nu", "else"
# ============================================================

R = DT("8.314")
T = DT("1000.0")

phi = DT("1.0")
psi = DT("1.0")

# --- CHANGE THIS ONLY ---
case_map = {"AB_XX": "gamma", "AB_YY": "gamma"}
# case_map = {"AB_XX": "nu",    "AB_YY": "nu"}
# case_map = {"AB_XX": "else",  "AB_YY": "else"}

H_LIST = [DT("1e-2"), DT("1e-3"), DT("1e-4"), DT("1e-5"), DT("1e-6")]
SEEDS  = [0, 1]   # 1–2 random n0 points

p_exp = DT("1.0")
q_exp = DT("1.0")
r_exp = DT("3.0")   # modifier uses (r_exp-1)

cations = ["A","B","C"]
anions  = ["X","Y"]

def delta(a,b): return DT("1.0") if a==b else DT("0.0")

# -------------------------
# Build symmetric pairs and quadruplets
# -------------------------
cat_pairs = []
for ii,i in enumerate(cations):
    for j in cations[ii:]:
        cat_pairs.append((i,j))
an_pairs = [("X","X"),("X","Y"),("Y","Y")]

quad = []
names = []
for (i,j) in cat_pairs:
    for (k,l) in an_pairs:
        quad.append((i,j,k,l))
        names.append(f"{i}{j}_{k}{l}")

idx = {nm:i for i,nm in enumerate(names)}
M = len(names)
ONES = np.ones(M, dtype=DT)

# -------------------------
# Linear weights for ideal-term derived quantities
# -------------------------
w_site1 = {i: np.array([(delta(a,i)+delta(b,i))/DT("2.0") for (a,b,x,y) in quad], dtype=DT) for i in cations}
w_site2 = {k: np.array([(delta(x,k)+delta(y,k))/DT("2.0") for (a,b,x,y) in quad], dtype=DT) for k in anions}

w_pair = {(i,k): np.array([((delta(a,i)+delta(b,i))*(delta(x,k)+delta(y,k)))/DT("4.0")
                           for (a,b,x,y) in quad], dtype=DT)
          for i in cations for k in anions}

w_row   = {i: sum(w_pair[(i,k)] for k in anions) for i in cations}  # A_i
w_col   = {k: sum(w_pair[(i,k)] for i in cations) for k in anions}  # B_k
w_Npair = sum(w_pair[(i,k)] for i in cations for k in anions)       # should equal ONES

# -------------------------
# Helpers: ln derivatives of linear/ratio forms
# -------------------------
def ln_linear_derivs(w, n):
    n = asDT(n)
    L = np.dot(w, n)
    if L <= 0: raise ValueError("Nonpositive linear form inside log.")
    ln_p  = w / L
    ln_pq = -np.outer(w,w)/(L*L)
    return np.log(L), ln_p, ln_pq

def ln_ratio_derivs(wA, wB, n):
    n = asDT(n)
    A = np.dot(wA, n)
    B = np.dot(wB, n)
    if A <= 0 or B <= 0: raise ValueError("Nonpositive ratio inside log.")
    ln_p  = wA/A - wB/B
    ln_pq = -np.outer(wA,wA)/(A*A) + np.outer(wB,wB)/(B*B)
    return np.log(A)-np.log(B), ln_p, ln_pq

def ln_Xquad_derivs(r, n):
    n = asDT(n)
    n_r = n[r]; N = np.sum(n, dtype=DT)
    if n_r <= 0 or N <= 0: raise ValueError("n_r or Nquad nonpositive.")
    e = np.zeros(M, dtype=DT); e[r]=DT("1.0")
    ln_p  = e/n_r - ONES/N
    ln_pq = -np.outer(e,e)/(n_r*n_r) + np.outer(ONES,ONES)/(N*N)
    return np.log(n_r)-np.log(N), ln_p, ln_pq

# ============================================================
# IDEAL TERM: G_id and H_id
# ============================================================
def compute_state_id(n):
    n = asDT(n)
    if np.any(n<=0): raise ValueError("All n must be >0 for logs.")
    Nquad = np.sum(n, dtype=DT)

    n_site1 = {i: np.dot(w_site1[i], n) for i in cations}
    n_site2 = {k: np.dot(w_site2[k], n) for k in anions}
    # N1 = N2 = Nquad by construction of weights
    X_site1 = {i: n_site1[i]/Nquad for i in cations}
    X_site2 = {k: n_site2[k]/Nquad for k in anions}

    n_pair = {(i,k): np.dot(w_pair[(i,k)], n) for i in cations for k in anions}
    Npair  = sum(n_pair.values())
    X_pair = {(i,k): n_pair[(i,k)]/Npair for i in cations for k in anions}

    A_row = {i: sum(n_pair[(i,k)] for k in anions) for i in cations}
    B_col = {k: sum(n_pair[(i,k)] for i in cations) for k in anions}

    # Y-block (entropy): Y_i^(1st)=n_i/Nquad, Y_k^(2nd)=n_k/Nquad
    Y1 = {i: n_site1[i]/Nquad for i in cations}
    Y2 = {k: n_site2[k]/Nquad for k in anions}

    return dict(Nquad=Nquad, Npair=Npair,
                n_site1=n_site1, n_site2=n_site2,
                X_site1=X_site1, X_site2=X_site2,
                n_pair=n_pair, X_pair=X_pair,
                A_row=A_row, B_col=B_col,
                Y1=Y1, Y2=Y2)

def G_id(n):
    st = compute_state_id(n)

    # S1
    S1 = DT("0.0")
    for i in cations:
        S1 += st["n_site1"][i] * np.log(st["X_site1"][i])
    for k in anions:
        S1 += st["n_site2"][k] * np.log(st["X_site2"][k])

    # S2 using expanded log: ln(n_{i/k}) + ln(Npair) - ln(A_i) - ln(B_k)
    S2 = DT("0.0")
    Npair = st["Npair"]
    for i in cations:
        Ai = st["A_row"][i]
        for k in anions:
            nik = st["n_pair"][(i,k)]
            Bk  = st["B_col"][k]
            S2 += nik * (np.log(nik) + np.log(Npair) - np.log(Ai) - np.log(Bk))

    # S3
    S3 = DT("0.0")
    Nquad = st["Nquad"]
    for r,(i,j,k,l) in enumerate(quad):
        nr = n[r]
        Xr = nr/Nquad
        C  = (DT("2.0") - (DT("1.0") if i==j else DT("0.0")))*(DT("2.0") - (DT("1.0") if k==l else DT("0.0")))
        ln_u = np.log(Xr) - np.log(C)

        pairs = [(i,k),(i,l),(j,k),(j,l)]
        ln_u -= phi * sum(np.log(st["X_pair"][pk]) for pk in pairs)

        ln_u += psi*( np.log(st["Y1"][i]) + np.log(st["Y1"][j])
                    + np.log(st["Y2"][k]) + np.log(st["Y2"][l]) )
        S3 += nr*ln_u

    return R*T*(S1+S2+S3)

def H_id_analytic(n):
    n = asDT(n)
    compute_state_id(n)

    H = np.zeros((M,M), dtype=DT)

    _, lnNpair_p, lnNpair_pq = ln_linear_derivs(w_Npair, n)

    lnRow_p, lnRow_pq = {}, {}
    for i in cations:
        _, lp, lpq = ln_linear_derivs(w_row[i], n)
        lnRow_p[i], lnRow_pq[i] = lp, lpq

    lnCol_p, lnCol_pq = {}, {}
    for k in anions:
        _, lp, lpq = ln_linear_derivs(w_col[k], n)
        lnCol_p[k], lnCol_pq[k] = lp, lpq

    lnPair_p, lnPair_pq = {}, {}
    for i in cations:
        for k in anions:
            _, lp, lpq = ln_linear_derivs(w_pair[(i,k)], n)
            lnPair_p[(i,k)], lnPair_pq[(i,k)] = lp, lpq

    # S1: f = n_i ln(n_i/Nquad) for both sublattices
    for i in cations:
        wN = w_site1[i]
        Ni = np.dot(wN, n)
        _, ln_p, ln_pq = ln_ratio_derivs(w_site1[i], ONES, n)
        H += np.outer(wN, ln_p) + np.outer(ln_p, wN) + Ni*ln_pq

    for k in anions:
        wN = w_site2[k]
        Nk = np.dot(wN, n)
        _, ln_p, ln_pq = ln_ratio_derivs(w_site2[k], ONES, n)
        H += np.outer(wN, ln_p) + np.outer(ln_p, wN) + Nk*ln_pq

    # S2: f = n_{i/k} ln(n_{i/k} Npair / (A_i B_k))
    for i in cations:
        for k in anions:
            wN = w_pair[(i,k)]
            Nik = np.dot(wN, n)
            ln_u_p  = lnPair_p[(i,k)]  + lnNpair_p  - lnRow_p[i]  - lnCol_p[k]
            ln_u_pq = lnPair_pq[(i,k)] + lnNpair_pq - lnRow_pq[i] - lnCol_pq[k]
            H += np.outer(wN, ln_u_p) + np.outer(ln_u_p, wN) + Nik*ln_u_pq

    # S3 blocks
    lnXpair_p  = {(i,k): lnPair_p[(i,k)]  - lnNpair_p  for i in cations for k in anions}
    lnXpair_pq = {(i,k): lnPair_pq[(i,k)] - lnNpair_pq for i in cations for k in anions}

    lnY1_p  = {i: ln_ratio_derivs(w_site1[i], ONES, n)[1] for i in cations}
    lnY1_pq = {i: ln_ratio_derivs(w_site1[i], ONES, n)[2] for i in cations}
    lnY2_p  = {k: ln_ratio_derivs(w_site2[k], ONES, n)[1] for k in anions}
    lnY2_pq = {k: ln_ratio_derivs(w_site2[k], ONES, n)[2] for k in anions}

    for r,(i,j,k,l) in enumerate(quad):
        nr = n[r]
        _, lnXr_p, lnXr_pq = ln_Xquad_derivs(r, n)

        ln_u_p  = lnXr_p.copy()
        ln_u_pq = lnXr_pq.copy()

        pairs = [(i,k),(i,l),(j,k),(j,l)]
        for pk in pairs:
            ln_u_p  -= phi * lnXpair_p[pk]
            ln_u_pq -= phi * lnXpair_pq[pk]

        ln_u_p  += psi*(lnY1_p[i]  + lnY1_p[j]  + lnY2_p[k]  + lnY2_p[l])
        ln_u_pq += psi*(lnY1_pq[i] + lnY1_pq[j] + lnY2_pq[k] + lnY2_pq[l])

        H[r,:] += ln_u_p
        H[:,r] += ln_u_p
        H += nr * ln_u_pq

    return R*T*H

# ============================================================
# EXCESS TERM: Y_{m/k}, xi-form base + ternary modifier cases
# ============================================================
def w_vec_Y(m, k):
    # weight for two-index Y_{m/k} using quadruplet fraction definition
    w = np.zeros(M, dtype=DT)
    for p,(i,j,x,y) in enumerate(quad):
        w[p] = ((delta(i,m)+delta(j,m))*(delta(x,k)+delta(y,k)))/4.0
    return w

W_Y = {(m,k): w_vec_Y(m,k) for m in cations for k in anions}

def Ymk_and_derivs(n, m, k):
    # Y = (w·n)/N, N=sum(n)
    n = asDT(n)
    N = np.sum(n, dtype=DT)
    w = W_Y[(m,k)]
    n = asDT(n)
    A = np.dot(w,n)
    Y = A/N
    Y_p  = (w - Y)/N
    Y_pq = -(Y_p[:,None] + Y_p[None,:]) / N
    return Y, Y_p, Y_pq

def ln_derivs(z, z_p, z_pq):
    if z <= 0: raise ValueError("log argument <= 0")
    ln_p  = z_p / z
    ln_pq = z_pq / z - np.outer(z_p,z_p)/(z*z)
    return ln_p, ln_pq

def ratio_derivs(y, y_p, y_pq, d, d_p, d_pq):
    # w = y/d
    w    = y/d
    w_p  = (y_p*d - y*d_p)/(d*d)
    w_pq = (y_pq*d - np.outer(y_p,d_p) - np.outer(d_p,y_p) - y*d_pq)/(d*d) + 2*y*np.outer(d_p,d_p)/(d**3)
    return w, w_p, w_pq

# deterministic-ish coefficients for diagonal terms
gcoeff = {}
for (i,j) in cat_pairs:
    for kk in ["XX","YY"]:
        gcoeff[f"{i}{j}_{kk}"] = DT(str(1500.0 + 100.0*((hash(f"{i}{j}_{kk}") % 11) - 5)))

def xi_for_AB_case(k, case, YA, YA_p, YA_pq, YB, YB_p, YB_pq, YC, YC_p, YC_pq):
    """
    Make nu/gamma disjoint so "else" stays valid:
      gamma: xi_ij = YA            , xi_ji = YB + YC
      nu:    xi_ij = YA + YC       , xi_ji = YB
      else:  xi_ij = YA            , xi_ji = YB
    """
    if case == "gamma":
        a, a_p, a_pq = YA, YA_p, YA_pq
        b, b_p, b_pq = (YB+YC), (YB_p+YC_p), (YB_pq+YC_pq)
    elif case == "nu":
        a, a_p, a_pq = (YA+YC), (YA_p+YC_p), (YA_pq+YC_pq)
        b, b_p, b_pq = YB, YB_p, YB_pq
    elif case == "else":
        a, a_p, a_pq = YA, YA_p, YA_pq
        b, b_p, b_pq = YB, YB_p, YB_pq
    else:
        raise ValueError("case must be gamma/nu/else")
    return a,a_p,a_pq,b,b_p,b_pq

def delta_g_and_derivs_ex(n, case_map):
    n = asDT(n)
    if np.any(n<=0): raise ValueError("All n must be >0.")

    dg    = np.zeros(M, dtype=DT)
    dg_p  = np.zeros((M,M), dtype=DT)
    dg_pq = np.zeros((M,M,M), dtype=DT)

    # Precompute Y_{m/k} for m in {A,B,C}, k in {X,Y}
    Y = {}; Yp = {}; Ypq = {}
    for m in cations:
        for k in anions:
            val, val_p, val_pq = Ymk_and_derivs(n,m,k)
            Y[(m,k)], Yp[(m,k)], Ypq[(m,k)] = val, val_p, val_pq

    def fill_one(rname, i, j, k):
        r  = idx[rname]
        gr = gcoeff[rname]

        YA, YA_p, YA_pq = Y[("A",k)], Yp[("A",k)], Ypq[("A",k)]
        YB, YB_p, YB_pq = Y[("B",k)], Yp[("B",k)], Ypq[("B",k)]
        YC, YC_p, YC_pq = Y[("C",k)], Yp[("C",k)], Ypq[("C",k)]

        # choose xi
        if (i,j) == ("A","B"):
            case = case_map.get(rname, "else")  # default to else if not specified
            a,a_p,a_pq,b,b_p,b_pq = xi_for_AB_case(k, case, YA,YA_p,YA_pq, YB,YB_p,YB_pq, YC,YC_p,YC_pq)
        else:
            # binary-style xi: xi_ij = Y_{i/k}, xi_ji = Y_{j/k}
            a,a_p,a_pq = Y[(i,k)], Yp[(i,k)], Ypq[(i,k)]
            b,b_p,b_pq = Y[(j,k)], Yp[(j,k)], Ypq[(j,k)]

        s    = a + b
        s_p  = a_p + b_p
        s_pq = a_pq + b_pq
        if a<=0 or b<=0 or s<=0: raise ValueError("Need positive xi a,b,s for logs.")

        # base = a^p b^q / s^(p+q)
        base = (a**p_exp)*(b**q_exp)/(s**(p_exp+q_exp))

        # ternary modifier for AB_XX/AB_YY only
        case = case_map.get(rname, "none")
        Mval = DT("1.0")
        if case != "none":
            # use m=C (explicit ternary)
            Ym, Ym_p, Ym_pq = YC, YC_p, YC_pq
            if case == "gamma":
                # M = (Yc/b) * (1 - Yb/b)^(r-1)
                w, w_p, w_pq = ratio_derivs(YB, YB_p, YB_pq, b, b_p, b_pq)  # w=Yb/b
                t    = DT("1.0") - w
                t_p  = -w_p
                t_pq = -w_pq
                if Ym <= 0 or t <= 0: raise ValueError("gamma needs Yc>0 and (1-Yb/b)>0")
                Mval = (Ym/b) * (t**(r_exp-1.0))
            elif case == "nu":
                # M = (Yc/a) * (1 - Ya/a)^(r-1)   (note Ya corresponds to i=A)
                w, w_p, w_pq = ratio_derivs(YA, YA_p, YA_pq, a, a_p, a_pq)   # w=Ya/a
                t    = DT("1.0") - w
                t_p  = -w_p
                t_pq = -w_pq
                if Ym <= 0 or t <= 0: raise ValueError("nu needs Yc>0 and (1-Ya/a)>0")
                Mval = (Ym/a) * (t**(r_exp-1.0))
            elif case == "else":
                # M = Yc * (1 - a - b)^(r-1)
                t    = DT("1.0") - a - b
                t_p  = -(a_p + b_p)
                t_pq = -(a_pq + b_pq)
                if Ym <= 0 or t <= 0: raise ValueError("else needs Yc>0 and (1-a-b)>0")
                Mval = Ym * (t**(r_exp-1.0))
            else:
                raise ValueError("case must be gamma/nu/else/none")

        val = gr * base * Mval
        dg[r] = val

        # log-derivatives of base
        ln_a_p, ln_a_pq = ln_derivs(a, a_p, a_pq)
        ln_b_p, ln_b_pq = ln_derivs(b, b_p, b_pq)
        ln_s_p, ln_s_pq = ln_derivs(s, s_p, s_pq)

        Lam_p  = p_exp*ln_a_p + q_exp*ln_b_p - (p_exp+q_exp)*ln_s_p
        Lam_pq = p_exp*ln_a_pq + q_exp*ln_b_pq - (p_exp+q_exp)*ln_s_pq

        # log-derivatives of modifier (if active)
        if case != "none":
            Ym, Ym_p, Ym_pq = YC, YC_p, YC_pq
            ln_Ym_p, ln_Ym_pq = ln_derivs(Ym, Ym_p, Ym_pq)

            if case == "gamma":
                w, w_p, w_pq = ratio_derivs(YB, YB_p, YB_pq, b, b_p, b_pq)
                t, t_p, t_pq = (1.0-w), (-w_p), (-w_pq)
                ln_t_p, ln_t_pq = ln_derivs(t, t_p, t_pq)
                ln_b_p, ln_b_pq = ln_derivs(b, b_p, b_pq)
                Lam_p  = Lam_p  + ln_Ym_p  - ln_b_p  + (r_exp-1.0)*ln_t_p
                Lam_pq = Lam_pq + ln_Ym_pq - ln_b_pq + (r_exp-1.0)*ln_t_pq

            elif case == "nu":
                w, w_p, w_pq = ratio_derivs(YA, YA_p, YA_pq, a, a_p, a_pq)
                t, t_p, t_pq = (1.0-w), (-w_p), (-w_pq)
                ln_t_p, ln_t_pq = ln_derivs(t, t_p, t_pq)
                ln_a_p, ln_a_pq = ln_derivs(a, a_p, a_pq)
                Lam_p  = Lam_p  + ln_Ym_p  - ln_a_p  + (r_exp-1.0)*ln_t_p
                Lam_pq = Lam_pq + ln_Ym_pq - ln_a_pq + (r_exp-1.0)*ln_t_pq

            elif case == "else":
                t    = DT("1.0") - a - b
                t_p  = -(a_p + b_p)
                t_pq = -(a_pq + b_pq)
                ln_t_p, ln_t_pq = ln_derivs(t, t_p, t_pq)
                Lam_p  = Lam_p  + ln_Ym_p  + (r_exp-1.0)*ln_t_p
                Lam_pq = Lam_pq + ln_Ym_pq + (r_exp-1.0)*ln_t_pq

        dg_p[r,:]    = val * Lam_p
        dg_pq[r,:,:] = val * (np.outer(Lam_p,Lam_p) + Lam_pq)

    # Fill only kk=XX and kk=YY terms; /XY dg = 0
    for (i,j) in cat_pairs:
        for kk in ["XX","YY"]:
            k = "X" if kk=="XX" else "Y"
            fill_one(f"{i}{j}_{kk}", i, j, k)

    return dg, dg_p, dg_pq

def P_Q_and_derivs_ex(n):
    """
    Eq.17 prefactors, Z=1:
      P_{ij/XX} = 0.5*n_{ij/XY},  P_{ij/YY} = 0.5*n_{ij/XY}
      Q_{ii/an} = 0.5*sum_{m!=i} n_{im/an}
    """
    n = asDT(n)
    P  = np.zeros(M, dtype=DT); Q  = np.zeros(M, dtype=DT)
    Pp = np.zeros((M,M), dtype=DT); Qp = np.zeros((M,M), dtype=DT)

    # P
    for (i,j) in cat_pairs:
        ixy = idx[f"{i}{j}_XY"]
        for kk in ["XX","YY"]:
            r = idx[f"{i}{j}_{kk}"]
            P[r] = DT("0.5")*n[ixy]
            Pp[r,ixy] = DT("0.5")

    # Q
    for i in cations:
        for an in ["XX","XY","YY"]:
            r = idx[f"{i}{i}_{an}"]
            s = DT("0.0")
            for m in cations:
                if m==i: continue
                a,b = (i,m) if i<=m else (m,i)
                jidx = idx[f"{a}{b}_{an}"]
                s += n[jidx]
                Qp[r,jidx] += DT("0.5")
            Q[r] = DT("0.5")*s

    return P,Q,Pp,Qp

def G_ex(n, case_map):
    n = asDT(n)
    dg,_,_ = delta_g_and_derivs_ex(n, case_map)
    P,Q,_,_ = P_Q_and_derivs_ex(n)

    T1 = np.dot(n, dg)

    diag_l_eq_k = [idx[nm] for nm in names if (nm.endswith("_XX") or nm.endswith("_YY"))]
    T2 = np.dot(P[diag_l_eq_k], dg[diag_l_eq_k])

    diag_j_eq_i = [idx[nm] for nm in names if nm[0]==nm[1]]
    T3 = np.dot(Q[diag_j_eq_i], dg[diag_j_eq_i])

    return DT("0.5")*(T1+T2+T3)

def H_ex_analytic(n, case_map):
    n = asDT(n)
    dg,dg_p,dg_pq = delta_g_and_derivs_ex(n, case_map)
    P,Q,Pp,Qp = P_Q_and_derivs_ex(n)

    H = np.zeros((M,M), dtype=DT)
    diag_l_eq_k = [idx[nm] for nm in names if (nm.endswith("_XX") or nm.endswith("_YY"))]
    diag_j_eq_i = [idx[nm] for nm in names if nm[0]==nm[1]]

    for p in range(M):
        for q in range(M):
            term = DT("0.0")
            term += dg_p[p,q] + dg_p[q,p]
            term += np.dot(n, dg_pq[:,p,q])
            for r in diag_l_eq_k:
                term += Pp[r,p]*dg_p[r,q] + Pp[r,q]*dg_p[r,p] + P[r]*dg_pq[r,p,q]
            for r in diag_j_eq_i:
                term += Qp[r,p]*dg_p[r,q] + Qp[r,q]*dg_p[r,p] + Q[r]*dg_pq[r,p,q]
            H[p,q] = DT("0.5")*term
    return H

# ============================================================
# TOTAL + FD
# ============================================================
def G_tot(n, case_map):
    return G_id(n) + G_ex(n, case_map)

def H_tot_analytic(n, case_map):
    return H_id_analytic(n) + H_ex_analytic(n, case_map)

def H_fd_total(n0, h, case_map):
    n0 = asDT(n0)
    h = DT(h)
    if np.min(n0) <= h:
        raise ValueError("h too large; would make some n negative.")
    H = np.zeros((M,M), dtype=DT)
    G0 = G_tot(n0, case_map)

    for p in range(M):
        e = np.zeros(M, dtype=DT); e[p]=DT("1.0")
        H[p,p] = (G_tot(n0+h*e, case_map) - DT("2.0")*G0 + G_tot(n0-h*e, case_map))/(h*h)

    for p in range(M):
        ep = np.zeros(M, dtype=DT); ep[p]=DT("1.0")
        for q in range(p+1,M):
            eq = np.zeros(M, dtype=DT); eq[q]=DT("1.0")
            Gpp = G_tot(n0+h*ep+h*eq, case_map)
            Gpm = G_tot(n0+h*ep-h*eq, case_map)
            Gmp = G_tot(n0-h*ep+h*eq, case_map)
            Gmm = G_tot(n0-h*ep-h*eq, case_map)
            val = (Gpp - Gpm - Gmp + Gmm)/(DT("4.0")*h*h)
            H[p,q]=val; H[q,p]=val
    return H

def random_valid_n0(seed, case_map, tries=200):
    rng = np.random.default_rng(seed)
    for t in range(tries):
        n0 = asDT(0.8 + rng.random(M))  # positive, not tiny
        try:
            _ = G_tot(n0, case_map)
            return n0
        except Exception:
            continue
    raise RuntimeError("Could not find a valid n0 (try changing seed or ranges).")

def fro_norm(M):
    M = asDT(M)
    return np.sqrt(np.sum(M * M, dtype=DT))

# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    print("M =", M)
    print("case_map =", case_map)
    print("Example vars:", names[:12], "...\n")

    for seed in SEEDS:
        n0 = random_valid_n0(seed, case_map)

        Ha = H_tot_analytic(n0, case_map)
        sym = fro_norm(Ha-Ha.T)/max(1.0, fro_norm(Ha))

        print("="*70)
        print("seed =", seed)
        print("G_id  =", float(G_id(n0)))
        print("G_ex  =", float(G_ex(n0, case_map)))
        print("G_tot =", float(G_tot(n0, case_map)))
        print("||H_tot analytic||_F =", float(fro_norm(Ha)))
        print("analytic symmetry err =", f"{float(sym):.3e}")

        best = (None, DT("1e99"))
        for h in H_LIST:
            if np.min(n0) <= h:
                print("skip h", h)
                continue
            Hfd = H_fd_total(n0, h, case_map)
            rel = fro_norm(Hfd - Ha)/max(1.0, fro_norm(Ha))
            print(f"h={float(h):g}  rel_err={float(rel):.3e}")
            if rel < best[1]:
                best = (h, rel)
        print("best ~", best)

M = 18
case_map = {'AB_XX': 'gamma', 'AB_YY': 'gamma'}
Example vars: ['AA_XX', 'AA_XY', 'AA_YY', 'AB_XX', 'AB_XY', 'AB_YY', 'AC_XX', 'AC_XY', 'AC_YY', 'BB_XX', 'BB_XY', 'BB_YY'] ...

seed = 0
G_id  = -332230.5094098724
G_ex  = 5275.1279853608385
G_tot = -326955.3814245116
||H_tot analytic||_F = 24889.821518539604
analytic symmetry err = 0.000e+00
h=0.01  rel_err=2.178e-05
h=0.001  rel_err=2.178e-07
h=0.0001  rel_err=2.447e-09
h=1e-05  rel_err=2.726e-07
h=1e-06  rel_err=1.850e-05
best ~ (np.longdouble('0.0001'), np.longdouble('2.446795826215609672e-09'))
seed = 1
G_id  = -325532.8951043893
G_ex  = 5096.826909433423
G_tot = -320436.0681949559
||H_tot analytic||_F = 24099.95910901714
analytic symmetry err = 0.000e+00
h=0.01  rel_err=1.744e-05
h=0.001  rel_err=1.744e-07
h=0.0001  rel_err=3.600e-09
h=1e-05  rel_err=2.638e-07
h=1e-06  rel_err=1.984e-05
best ~ (np.longdouble('0.0001'), np.longdouble('3.5996037587011411132e-09'))
